# Severstal Stage-1 Anomaly Detector Benchmark (Colab)

This notebook benchmarks anomaly detectors for **two-stage layer-1** selection on Severstal.

Primary baseline:
- `patchcore_repo_baseline`

Optional external models (official repos only):
- `rd4ad_knn_stage1`
- `cflow_encoder_knn_stage1`
- `simplenet_stage1` (best-effort, API-dependent)

All models are compared on the same pilot split and evaluated at fixed `FPR_normal` operating points.


In [ ]:
# 1) Runtime preflight
import sys
import numpy as np
import torch
import torchvision

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('TorchVision:', torchvision.__version__)

if not np.__version__.startswith('1.26'):
    raise RuntimeError(
        'Need numpy 1.26.x for stable sklearn/torch interop in this notebook. '
        'Install numpy==1.26.4, restart runtime, then rerun.'
    )

print('Preflight OK.')


In [ ]:
# 2) Repo setup
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/content/FYP-code')
if not REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/spinelessknave8/FYP_code.git', str(REPO)])

subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'])
os.chdir(REPO)

for p in [
    str(REPO),
    str(REPO / 'severstral-osr' / 'src'),
    str(REPO / 'src'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Repo ready:', REPO)


In [ ]:
# 3) Drive + dataset checks
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

DATA_ROOT = Path('/content/drive/MyDrive/datasets/severstal')
TRAIN_CSV = DATA_ROOT / 'train.csv'
TRAIN_IMAGES = DATA_ROOT / 'train_images'
OUT_ROOT = Path('/content/drive/MyDrive/fyp_outputs/severstral_stage1_ad_benchmark')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

assert TRAIN_CSV.exists(), f'Missing: {TRAIN_CSV}'
assert TRAIN_IMAGES.exists() and TRAIN_IMAGES.is_dir(), f'Missing: {TRAIN_IMAGES}'

print('DATA_ROOT:', DATA_ROOT)
print('OUT_ROOT :', OUT_ROOT)


In [ ]:
# 4) Build shared pilot split (normal train/val/test + defect test)
import csv
import json
import random
from collections import defaultdict
from pathlib import Path

SEED = 42
rng = random.Random(SEED)

# Read defect image ids from CSV (any positive mask row means defect image)
defect_ids = set()
with open(TRAIN_CSV, 'r', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        enc = str(row.get('EncodedPixels', '')).strip().lower()
        if enc and enc != 'nan':
            defect_ids.add(row['ImageId'].strip())

all_images = [p for p in TRAIN_IMAGES.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}]
normal_paths = [str(p) for p in all_images if p.name not in defect_ids]
defect_paths = [str(TRAIN_IMAGES / img_id) for img_id in sorted(defect_ids) if (TRAIN_IMAGES / img_id).exists()]

rng.shuffle(normal_paths)
rng.shuffle(defect_paths)

# Keep pilot small enough for Colab iteration
MAX_NORMAL_TRAIN = 1200
MAX_NORMAL_VAL = 300
MAX_NORMAL_TEST = 300
MAX_DEFECT_TEST = 800

normal_train = normal_paths[:MAX_NORMAL_TRAIN]
normal_val = normal_paths[MAX_NORMAL_TRAIN:MAX_NORMAL_TRAIN + MAX_NORMAL_VAL]
normal_test = normal_paths[MAX_NORMAL_TRAIN + MAX_NORMAL_VAL:MAX_NORMAL_TRAIN + MAX_NORMAL_VAL + MAX_NORMAL_TEST]
defect_test = defect_paths[:MAX_DEFECT_TEST]

if min(len(normal_train), len(normal_val), len(normal_test), len(defect_test)) == 0:
    raise RuntimeError('Split is empty; check dataset path/content or increase caps.')

split = {
    'seed': SEED,
    'normal_train': normal_train,
    'normal_val': normal_val,
    'normal_test': normal_test,
    'defect_test': defect_test,
    'counts': {
        'normal_train': len(normal_train),
        'normal_val': len(normal_val),
        'normal_test': len(normal_test),
        'defect_test': len(defect_test),
    },
}

split_path = OUT_ROOT / 'split_manifest.json'
split_path.write_text(json.dumps(split, indent=2))
print('Split saved:', split_path)
print('Counts:', split['counts'])


In [ ]:
# 5) Shared evaluation helpers
import time
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

FPR_TARGETS = [0.05, 0.10, 0.20]


def evaluate_stage1_scores(model_name, s_normal_val, s_normal_test, s_defect_test, runtime_sec=None, extra=None):
    y = np.concatenate([np.zeros(len(s_normal_test)), np.ones(len(s_defect_test))])
    s = np.concatenate([s_normal_test, s_defect_test])
    auroc = float(roc_auc_score(y, s))

    rows = []
    for fpr in FPR_TARGETS:
        tau = float(np.quantile(s_normal_val, 1.0 - fpr))
        row = {
            'model': model_name,
            'fpr_target': fpr,
            'threshold': tau,
            'AUROC_defect_screening': auroc,
            'TPR_defect@FPR': float(np.mean(s_defect_test > tau)),
            'FPR_normal_realized': float(np.mean(s_normal_test > tau)),
        }
        if runtime_sec is not None:
            row['runtime_sec'] = float(runtime_sec)
        if extra:
            row.update(extra)
        rows.append(row)
    return pd.DataFrame(rows)


def append_results(df, csv_path):
    csv_path = Path(csv_path)
    if csv_path.exists():
        prev = pd.read_csv(csv_path)
        merged = pd.concat([prev, df], ignore_index=True)
        merged = merged.drop_duplicates(subset=['model', 'fpr_target'], keep='last')
    else:
        merged = df.copy()
    merged.to_csv(csv_path, index=False)
    return merged


RESULTS_CSV = OUT_ROOT / 'stage1_model_comparison.csv'
print('Results CSV:', RESULTS_CSV)


In [ ]:
# 6) PatchCore repo baseline (primary stage-1 baseline)
import json
import time
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from src.patchcore.patchcore_simple import (
    PatchCoreBackbone,
    build_memory_bank,
    coreset_subsample,
    infer_anomaly_scores,
)

split = json.loads((OUT_ROOT / 'split_manifest.json').read_text())
normal_train = split['normal_train']
normal_val = split['normal_val']
normal_test = split['normal_test']
defect_test = split['defect_test']

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class PathDS(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return tf(Image.open(self.paths[i]).convert('RGB'))

batch_size = 16
num_workers = 2
device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_loader = DataLoader(PathDS(normal_train), batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(PathDS(normal_val), batch_size=batch_size, shuffle=False, num_workers=num_workers)
normal_test_loader = DataLoader(PathDS(normal_test), batch_size=batch_size, shuffle=False, num_workers=num_workers)
defect_test_loader = DataLoader(PathDS(defect_test), batch_size=batch_size, shuffle=False, num_workers=num_workers)

cache_dir = OUT_ROOT / 'patchcore_cache'
cache_dir.mkdir(parents=True, exist_ok=True)
memory_path = cache_dir / 'memory.npy'

start = time.time()
backbone = PatchCoreBackbone('resnet18').to(device)

if memory_path.exists():
    memory = np.load(memory_path)
else:
    memory_pre = build_memory_bank(train_loader, device, backbone, num_patches_per_sample=128)
    memory = coreset_subsample(memory_pre, ratio=0.1, seed=42)
    np.save(memory_path, memory)

s_normal_val = infer_anomaly_scores(val_loader, device, backbone, memory)
s_normal_test = infer_anomaly_scores(normal_test_loader, device, backbone, memory)
s_defect_test = infer_anomaly_scores(defect_test_loader, device, backbone, memory)
runtime = time.time() - start

np.save(OUT_ROOT / 'scores_patchcore_normal_val.npy', s_normal_val)
np.save(OUT_ROOT / 'scores_patchcore_normal_test.npy', s_normal_test)
np.save(OUT_ROOT / 'scores_patchcore_defect_test.npy', s_defect_test)

df_patchcore = evaluate_stage1_scores(
    model_name='patchcore_repo_baseline',
    s_normal_val=s_normal_val,
    s_normal_test=s_normal_test,
    s_defect_test=s_defect_test,
    runtime_sec=runtime,
    extra={'notes': 'repo src.patchcore.patchcore_simple'},
)

all_results = append_results(df_patchcore, RESULTS_CSV)
print('PatchCore done. Rows now:', len(all_results))
display(df_patchcore)


In [ ]:
# 7) RD4AD official repo (optional): install + smoke + stage-1 kNN scoring
import json
import time
import numpy as np
import torch
import subprocess
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.neighbors import NearestNeighbors

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.7', 'opencv-python-headless==4.8.1.78', 'tqdm'])
    if not Path('/content/RD4AD').exists():
        subprocess.check_call(['git', 'clone', 'https://github.com/hq-deng/RD4AD', '/content/RD4AD'])

    if '/content/RD4AD' not in sys.path:
        sys.path.insert(0, '/content/RD4AD')

    from resnet import wide_resnet50_2

    split = json.loads((OUT_ROOT / 'split_manifest.json').read_text())

    tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])

    class PathDS(Dataset):
        def __init__(self, paths):
            self.paths = paths
        def __len__(self):
            return len(self.paths)
        def __getitem__(self, i):
            return tf(Image.open(self.paths[i]).convert('RGB'))

    def embed(paths, encoder, bs=16):
        dl = DataLoader(PathDS(paths), batch_size=bs, shuffle=False, num_workers=2)
        out = []
        with torch.no_grad():
            for x in dl:
                x = x.to(device)
                feats = encoder(x)
                z = feats[-1].mean(dim=(2,3))
                out.append(z.cpu().numpy())
        return np.concatenate(out, axis=0)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    start = time.time()
    encoder, bn = wide_resnet50_2(pretrained=True)
    encoder = encoder.to(device).eval()

    E_train = embed(split['normal_train'], encoder)
    E_val = embed(split['normal_val'], encoder)
    E_nte = embed(split['normal_test'], encoder)
    E_dte = embed(split['defect_test'], encoder)

    knn = NearestNeighbors(n_neighbors=5, metric='euclidean').fit(E_train)
    score = lambda E: knn.kneighbors(E, return_distance=True)[0].mean(axis=1)

    s_normal_val = score(E_val)
    s_normal_test = score(E_nte)
    s_defect_test = score(E_dte)
    runtime = time.time() - start

    np.save(OUT_ROOT / 'scores_rd4ad_normal_val.npy', s_normal_val)
    np.save(OUT_ROOT / 'scores_rd4ad_normal_test.npy', s_normal_test)
    np.save(OUT_ROOT / 'scores_rd4ad_defect_test.npy', s_defect_test)

    df_rd4ad = evaluate_stage1_scores(
        model_name='rd4ad_knn_stage1',
        s_normal_val=s_normal_val,
        s_normal_test=s_normal_test,
        s_defect_test=s_defect_test,
        runtime_sec=runtime,
        extra={'notes': 'official RD4AD encoder features + kNN scorer'},
    )
    all_results = append_results(df_rd4ad, RESULTS_CSV)
    print('RD4AD done. Rows now:', len(all_results))
    display(df_rd4ad)
except Exception as e:
    print('RD4AD section skipped due to error:')
    print(type(e).__name__, e)


In [ ]:
# 8) CFLOW-AD official repo (optional): install + encoder kNN stage-1 scoring
import json
import time
import numpy as np
import torch
import subprocess
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.neighbors import NearestNeighbors

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'FrEIA==0.2', 'timm==0.9.7'])
    if not Path('/content/cflow-ad').exists():
        subprocess.check_call(['git', 'clone', 'https://github.com/gudovskiy/cflow-ad', '/content/cflow-ad'])

    if '/content/cflow-ad' not in sys.path:
        sys.path.insert(0, '/content/cflow-ad')

    from encoder import load_encoder_arch

    split = json.loads((OUT_ROOT / 'split_manifest.json').read_text())

    tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])

    class PathDS(Dataset):
        def __init__(self, paths):
            self.paths = paths
        def __len__(self):
            return len(self.paths)
        def __getitem__(self, i):
            return tf(Image.open(self.paths[i]).convert('RGB'))

    def embed(paths, encoder, bs=16):
        dl = DataLoader(PathDS(paths), batch_size=bs, shuffle=False, num_workers=2)
        out = []
        with torch.no_grad():
            for x in dl:
                x = x.to(device)
                feats = encoder(x)
                z = feats[-1].mean(dim=(2,3))
                out.append(z.cpu().numpy())
        return np.concatenate(out, axis=0)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    start = time.time()
    encoder = load_encoder_arch('wide_resnet50_2', 3)
    encoder = encoder.to(device).eval()

    E_train = embed(split['normal_train'], encoder)
    E_val = embed(split['normal_val'], encoder)
    E_nte = embed(split['normal_test'], encoder)
    E_dte = embed(split['defect_test'], encoder)

    knn = NearestNeighbors(n_neighbors=5, metric='euclidean').fit(E_train)
    score = lambda E: knn.kneighbors(E, return_distance=True)[0].mean(axis=1)

    s_normal_val = score(E_val)
    s_normal_test = score(E_nte)
    s_defect_test = score(E_dte)
    runtime = time.time() - start

    np.save(OUT_ROOT / 'scores_cflow_normal_val.npy', s_normal_val)
    np.save(OUT_ROOT / 'scores_cflow_normal_test.npy', s_normal_test)
    np.save(OUT_ROOT / 'scores_cflow_defect_test.npy', s_defect_test)

    df_cflow = evaluate_stage1_scores(
        model_name='cflow_encoder_knn_stage1',
        s_normal_val=s_normal_val,
        s_normal_test=s_normal_test,
        s_defect_test=s_defect_test,
        runtime_sec=runtime,
        extra={'notes': 'official CFLOW encoder features + kNN scorer'},
    )
    all_results = append_results(df_cflow, RESULTS_CSV)
    print('CFLOW done. Rows now:', len(all_results))
    display(df_cflow)
except Exception as e:
    print('CFLOW section skipped due to error:')
    print(type(e).__name__, e)


In [ ]:
# 9) SimpleNet official repo (optional): install + smoke/best-effort stage-1
import json
import time
import numpy as np
import torch
import subprocess
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.7'])
    if not Path('/content/SimpleNet').exists():
        subprocess.check_call(['git', 'clone', 'https://github.com/DonaldRR/SimpleNet', '/content/SimpleNet'])

    if '/content/SimpleNet' not in sys.path:
        sys.path.insert(0, '/content/SimpleNet')

    # API may differ by commit; this is best-effort.
    from simplenet import SimpleNet

    split = json.loads((OUT_ROOT / 'split_manifest.json').read_text())

    tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
    ])

    class PathDS(Dataset):
        def __init__(self, paths):
            self.paths = paths
        def __len__(self):
            return len(self.paths)
        def __getitem__(self, i):
            return tf(Image.open(self.paths[i]).convert('RGB'))

    def score_paths(paths, model, bs=16):
        dl = DataLoader(PathDS(paths), batch_size=bs, shuffle=False, num_workers=2)
        scores = []
        with torch.no_grad():
            for x in dl:
                x = x.to(device)
                out = model(x)
                out = out.reshape(len(x), -1).mean(dim=1)
                scores.append(out.detach().cpu().numpy())
        return np.concatenate(scores, axis=0)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    start = time.time()
    model = SimpleNet().to(device).eval()

    s_normal_val = score_paths(split['normal_val'], model)
    s_normal_test = score_paths(split['normal_test'], model)
    s_defect_test = score_paths(split['defect_test'], model)
    runtime = time.time() - start

    np.save(OUT_ROOT / 'scores_simplenet_normal_val.npy', s_normal_val)
    np.save(OUT_ROOT / 'scores_simplenet_normal_test.npy', s_normal_test)
    np.save(OUT_ROOT / 'scores_simplenet_defect_test.npy', s_defect_test)

    df_simplenet = evaluate_stage1_scores(
        model_name='simplenet_stage1',
        s_normal_val=s_normal_val,
        s_normal_test=s_normal_test,
        s_defect_test=s_defect_test,
        runtime_sec=runtime,
        extra={'notes': 'official SimpleNet forward score (best effort)'},
    )
    all_results = append_results(df_simplenet, RESULTS_CSV)
    print('SimpleNet done. Rows now:', len(all_results))
    display(df_simplenet)
except Exception as e:
    print('SimpleNet section skipped due to error:')
    print(type(e).__name__, e)


In [ ]:
# 10) Final comparison table + plots
import pandas as pd
import matplotlib.pyplot as plt

results = pd.read_csv(RESULTS_CSV)
results = results.sort_values(['model', 'fpr_target']).reset_index(drop=True)
display(results)

# AUROC ranking
auroc_df = results[['model', 'AUROC_defect_screening']].drop_duplicates().sort_values('AUROC_defect_screening', ascending=False)
plt.figure(figsize=(9, 4))
plt.bar(auroc_df['model'], auroc_df['AUROC_defect_screening'])
plt.ylim(0, 1)
plt.xticks(rotation=20, ha='right')
plt.title('Stage-1 AUROC (Normal vs Defect)')
plt.tight_layout()
plot1 = OUT_ROOT / 'plot_stage1_auroc_ranking.png'
plt.savefig(plot1, dpi=140)
plt.show()

# TPR_defect at FPR targets
pivot = results.pivot(index='model', columns='fpr_target', values='TPR_defect@FPR').sort_index()
plt.figure(figsize=(9, 4))
for fpr in sorted(pivot.columns):
    plt.plot(pivot.index, pivot[fpr], marker='o', label=f'FPR={int(fpr*100)}%')
plt.ylim(0, 1)
plt.xticks(rotation=20, ha='right')
plt.title('Stage-1 TPR_defect @ Fixed FPR_normal')
plt.legend()
plt.tight_layout()
plot2 = OUT_ROOT / 'plot_stage1_tpr_defect_vs_fpr.png'
plt.savefig(plot2, dpi=140)
plt.show()

print('Saved:')
print('-', RESULTS_CSV)
print('-', plot1)
print('-', plot2)


## 11) Rerun Guide (After Disconnect)

1. Reopen notebook and run cells top-to-bottom.
2. If `numpy` changed, reinstall `numpy==1.26.4`, then restart runtime.
3. PatchCore baseline should always run first.
4. External model cells are optional and isolated with `try/except`; failed cells do not block final comparison.
